# TellCo – User Analytics in the Telecommunication Industry
Notebook structure: **Setup → Task 1 Overview → Task 2 Engagement → Task 3 Experience → Task 4 Satisfaction**.
Run the cells from top to bottom. Change `DATA_PATH` in the first code cell to where your Excel file is saved.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 60)

## 1. Load the data

In [ ]:
DATA_PATH = 'telcom_data.xlsx'      # <-- change to your file name/path
raw = pd.read_excel(DATA_PATH)      # takes ~1 minute (150k rows x 55 columns)
print(raw.shape)
raw.tail(2)

The last row has no `Bearer Id`, no `Start` date, etc. – it is a **column-averages row** added at the bottom of the file, not a real session. We remove it, then tidy the column names.

In [ ]:
df = raw[raw['Start'].notna()].copy()          # drops the summary row
df.columns = df.columns.str.strip()
df = df.rename(columns={'MSISDN/Number': 'MSISDN'})
print(df.shape)

# Optional: save a fast copy so you don't wait for Excel every time
# df.to_pickle('telcom_clean_raw.pkl');  df = pd.read_pickle('telcom_clean_raw.pkl')

### 1.1 Data types & missing values

In [ ]:
df.dtypes.to_frame('dtype').T

In [ ]:
missing = df.isna().sum().to_frame('missing')
missing['pct'] = missing['missing'] / len(df) * 100
missing[missing['missing'] > 0].sort_values('pct', ascending=False)

### 1.2 Cleaning (missing values + outliers → replaced with the column mean)
* Rows without an `MSISDN` (customer number) can't be attributed to a user, so they are dropped.
* Numeric columns: values outside the IQR fences (Q1 − 1.5·IQR, Q3 + 1.5·IQR) and NaNs are replaced with the mean of the normal values.
* If a column has IQR = 0 (mostly zeros) the IQR rule would flag every non-zero value, so only NaNs are filled there.
* Handset text columns: missing → `'undefined'` (the dataset already uses that label).

In [ ]:
df = df.dropna(subset=['MSISDN']).copy()
df['MSISDN'] = df['MSISDN'].astype('int64')

apps = ['Social Media', 'Google', 'Email', 'Youtube', 'Netflix', 'Gaming', 'Other']
app_cols = [f'{a} {d} (Bytes)' for a in apps for d in ('DL', 'UL')]
num_cols = (['Dur. (ms)', 'Total DL (Bytes)', 'Total UL (Bytes)',
             'Avg RTT DL (ms)', 'Avg RTT UL (ms)',
             'Avg Bearer TP DL (kbps)', 'Avg Bearer TP UL (kbps)',
             'TCP DL Retrans. Vol (Bytes)', 'TCP UL Retrans. Vol (Bytes)'] + app_cols)

def count_outliers(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())

before = pd.DataFrame({'missing': df[num_cols].isna().sum(),
                       'outliers': [count_outliers(df[c]) for c in num_cols]})
before

In [ ]:
def treat_with_mean(frame, cols):
    frame = frame.copy()
    for c in cols:
        q1, q3 = frame[c].quantile([.25, .75]); iqr = q3 - q1
        if iqr == 0:
            inlier = frame[c].notna()                       # only fill NaN
        else:
            inlier = frame[c].between(q1 - 1.5*iqr, q3 + 1.5*iqr)   # NaN -> False
        frame.loc[~inlier, c] = frame.loc[inlier, c].mean()
    return frame

df = treat_with_mean(df, num_cols)
for c in ['Handset Type', 'Handset Manufacturer', 'Last Location Name']:
    df[c] = df[c].fillna('undefined')

print('Remaining missing values in cleaned columns:', int(df[num_cols].isna().sum().sum()))

# Task 1 – User Overview Analysis

### Top 10 handsets, top 3 manufacturers, top 5 handsets per manufacturer
('undefined' means the handset is unknown, so it is excluded from the rankings.)

In [ ]:
known = df[(df['Handset Type'] != 'undefined') & (df['Handset Manufacturer'] != 'undefined')]

top10_handsets = known['Handset Type'].value_counts().head(10)
top3_manufacturers = known['Handset Manufacturer'].value_counts().head(3)
print('Top 10 handsets'); display(top10_handsets)
print('Top 3 manufacturers'); display(top3_manufacturers)

top5_per_maker = {m: known.loc[known['Handset Manufacturer'] == m, 'Handset Type'].value_counts().head(5)
                  for m in top3_manufacturers.index}
for m, s in top5_per_maker.items():
    print(f'Top 5 handsets – {m}'); display(s)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
top10_handsets.sort_values().plot.barh(ax=ax[0], color='steelblue'); ax[0].set_title('Top 10 handsets')
top3_manufacturers.plot.bar(ax=ax[1], color='darkorange'); ax[1].set_title('Top 3 manufacturers')
plt.tight_layout(); plt.show()

**Interpretation & recommendation to marketing (write your own here):** e.g. which brands dominate the customer base, which handsets to target with bundles/offers, and which brand is under-represented.

## Task 1.1 – Aggregate per user

In [ ]:
for a in apps:
    df[f'{a} Total'] = df[f'{a} DL (Bytes)'] + df[f'{a} UL (Bytes)']

user = df.groupby('MSISDN').agg(
    sessions=('Bearer Id', 'count'),
    duration_ms=('Dur. (ms)', 'sum'),
    total_dl=('Total DL (Bytes)', 'sum'),
    total_ul=('Total UL (Bytes)', 'sum'),
    **{f'{a}_bytes': (f'{a} Total', 'sum') for a in apps})
user['total_data'] = user['total_dl'] + user['total_ul']
print(user.shape)
user.head()

In [ ]:
# Per-application DL and UL per user (if your report needs them separately)
user_app_dl_ul = df.groupby('MSISDN')[app_cols].sum()
user_app_dl_ul.head()

## Task 1.2 – Exploratory Data Analysis

**a) Variables and data types** (put this in a slide)

In [ ]:
user.dtypes.to_frame('dtype')

**b) Basic metrics** (mean, median, min, max …) and why they matter

In [ ]:
basic = user.describe().T
basic['median'] = user.median()
basic

**c) Non-graphical univariate analysis** – dispersion parameters

In [ ]:
disp = pd.DataFrame({
    'mean': user.mean(), 'median': user.median(),
    'variance': user.var(), 'std': user.std(),
    'range': user.max() - user.min(),
    'IQR': user.quantile(.75) - user.quantile(.25),
    'coef_of_variation': user.std() / user.mean()})
disp

**d) Graphical univariate analysis** – histograms (shape) and boxplots (spread/outliers)

In [ ]:
plot_cols = ['sessions', 'duration_ms', 'total_data'] + [f'{a}_bytes' for a in apps]
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, c in zip(axes.ravel(), plot_cols):
    sns.histplot(user[c], bins=40, ax=ax, color='steelblue'); ax.set_title(c)
for ax in axes.ravel()[len(plot_cols):]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, c in zip(axes.ravel(), plot_cols):
    sns.boxplot(y=user[c], ax=ax, color='lightcoral'); ax.set_title(c)
for ax in axes.ravel()[len(plot_cols):]: ax.axis('off')
plt.tight_layout(); plt.show()

**e) Bivariate analysis** – each application vs total (DL+UL) data

In [ ]:
app_total_cols = [f'{a}_bytes' for a in apps]
print(user[app_total_cols + ['total_data']].corr()['total_data'].drop('total_data').sort_values(ascending=False))

sample = user.sample(5000, random_state=42)          # sample only for plotting speed
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for ax, c in zip(axes.ravel(), app_total_cols):
    sns.regplot(x=sample[c], y=sample['total_data'], ax=ax,
                scatter_kws={'s': 5, 'alpha': .3}, line_kws={'color': 'red'})
    ax.set_title(f'{c} vs total_data')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

**f) Variable transformation** – decile classes based on total session duration

In [ ]:
user['decile'] = pd.qcut(user['duration_ms'].rank(method='first'), 10, labels=False) + 1   # 1 (shortest) ... 10 (longest)
decile_summary = user.groupby('decile').agg(users=('total_data', 'size'),
                                            total_duration_ms=('duration_ms', 'sum'),
                                            total_data=('total_data', 'sum'))
top5_deciles = decile_summary.loc[6:10]        # the top five decile classes
top5_deciles

In [ ]:
top5_deciles['total_data'].plot.bar(figsize=(8, 4), color='teal')
plt.title('Total DL+UL data per top-five decile class'); plt.ylabel('Bytes'); plt.show()

**g) Correlation analysis** – between application data volumes

In [ ]:
corr = user[app_total_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation between applications'); plt.show()

**h) Dimensionality reduction (PCA)**

In [ ]:
X = StandardScaler().fit_transform(user[app_total_cols])
pca = PCA().fit(X)
explained = pd.DataFrame({'PC': [f'PC{i+1}' for i in range(len(apps))],
                          'explained_var_%': pca.explained_variance_ratio_ * 100,
                          'cumulative_%': pca.explained_variance_ratio_.cumsum() * 100})
display(explained)
loadings = pd.DataFrame(pca.components_.T, index=app_total_cols, columns=explained['PC'])
display(loadings.iloc[:, :3])

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(apps)+1), pca.explained_variance_ratio_.cumsum()*100, marker='o')
plt.xlabel('Number of components'); plt.ylabel('Cumulative explained variance (%)'); plt.title('PCA')
plt.show()

**PCA interpretation (max 4 bullet points, write your own):** how many components are needed for ~80–90 % of the variance, which apps load on PC1, what that says about redundancy between apps.

# Task 2 – User Engagement Analysis

### 2.1 Engagement metrics per customer and top 10 per metric

In [ ]:
eng = user[['sessions', 'duration_ms', 'total_data']].copy()
for m in eng.columns:
    print(f'Top 10 customers by {m}')
    display(eng[m].nlargest(10).to_frame())

### Normalize + k-means (k = 3)

In [ ]:
eng_scaler = MinMaxScaler()
X_eng = eng_scaler.fit_transform(eng)

km_eng = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_eng)
eng['cluster'] = km_eng.labels_
eng['cluster'].value_counts().sort_index()

In [ ]:
cluster_stats = eng.groupby('cluster').agg(['min', 'max', 'mean', 'sum'])
cluster_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, m in zip(axes, ['sessions', 'duration_ms', 'total_data']):
    eng.groupby('cluster')[m].mean().plot.bar(ax=ax, color=['#4c72b0', '#dd8452', '#55a868'])
    ax.set_title(f'Average {m} per cluster')
plt.tight_layout(); plt.show()

sample = eng.sample(5000, random_state=1)
sns.scatterplot(data=sample, x='duration_ms', y='total_data', hue='cluster', palette='tab10', s=15)
plt.title('Engagement clusters'); plt.show()

**Interpretation (write your own):** describe each cluster – e.g. cluster with lowest sessions/duration/traffic = low engagement, etc. – and how many users are in each.

### Top 10 most engaged users per application

In [ ]:
for a in apps:
    print(f'Top 10 users – {a}')
    display(user[f'{a}_bytes'].nlargest(10).to_frame())

### Top 3 most used applications

In [ ]:
app_totals = user[app_total_cols].sum().sort_values(ascending=False)
app_totals.index = app_totals.index.str.replace('_bytes', '')
top3_apps = app_totals.head(3)
display(top3_apps.to_frame('total_bytes'))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top3_apps.plot.bar(ax=ax[0], color=['#4c72b0', '#dd8452', '#55a868']); ax[0].set_title('Top 3 applications by total traffic')
top3_apps.plot.pie(ax=ax[1], autopct='%1.1f%%'); ax[1].set_ylabel('')
plt.tight_layout(); plt.show()

Note: **'Other'** is a catch-all bucket, not a real app. If it ranks in the top 3, mention it and also report the top 3 with 'Other' excluded.

### Elbow method – optimal k

In [ ]:
inertia = []
ks = range(1, 11)
for k in ks:
    inertia.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_eng).inertia_)

plt.figure(figsize=(7, 4))
plt.plot(ks, inertia, marker='o'); plt.xlabel('k'); plt.ylabel('Inertia (within-cluster SSE)')
plt.title('Elbow method'); plt.show()

Look for the 'elbow' where the curve stops dropping sharply. Set that value below (the default 3 matches the Task 2.1 grouping).

In [ ]:
OPTIMAL_K = 3          # <-- update after reading the elbow plot
km_opt = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10).fit(X_eng)
eng['cluster_opt'] = km_opt.labels_
eng.groupby('cluster_opt')[['sessions', 'duration_ms', 'total_data']].mean()

# Task 3 – Experience Analytics

### 3.1 Aggregate per customer: avg TCP retransmission, avg RTT, handset type, avg throughput
(Missing/outliers were already replaced with the mean in the cleaning step; handset uses the **mode** per user.)

In [ ]:
df['tcp']  = df['TCP DL Retrans. Vol (Bytes)'] + df['TCP UL Retrans. Vol (Bytes)']
df['rtt']  = df['Avg RTT DL (ms)'] + df['Avg RTT UL (ms)']
df['tput'] = df['Avg Bearer TP DL (kbps)'] + df['Avg Bearer TP UL (kbps)']

exp = df.groupby('MSISDN').agg(avg_tcp=('tcp', 'mean'), avg_rtt=('rtt', 'mean'), avg_tput=('tput', 'mean'))

# most frequent (mode) handset per user, ignoring 'undefined'
known_sessions = df[df['Handset Type'] != 'undefined']
handset_mode = (known_sessions.groupby(['MSISDN', 'Handset Type']).size().reset_index(name='n')
                .sort_values(['MSISDN', 'n'], ascending=[True, False])
                .drop_duplicates('MSISDN').set_index('MSISDN')['Handset Type'])
exp['handset'] = handset_mode.reindex(exp.index).fillna('undefined')
exp.head()

### 3.2 Top 10, bottom 10 and most frequent 10 values

In [ ]:
def top_bottom_freq(series, name):
    out = pd.DataFrame({
        f'{name} top 10': series.nlargest(10).values,
        f'{name} bottom 10': series.nsmallest(10).values,
        f'{name} most frequent': series.value_counts().head(10).index.values,
        'frequency': series.value_counts().head(10).values})
    return out

display(top_bottom_freq(df['tcp'], 'TCP'))
display(top_bottom_freq(df['rtt'], 'RTT'))
display(top_bottom_freq(df['tput'], 'Throughput'))

### 3.3 Throughput and TCP retransmission per handset type
(Restricted to the 10 most common handsets so the plots stay readable.)

In [ ]:
exp_known = exp[exp['handset'] != 'undefined']
top_handsets = exp_known['handset'].value_counts().head(10).index
sub = exp_known[exp_known['handset'].isin(top_handsets)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=sub, y='handset', x='avg_tput', order=top_handsets, showfliers=False)
plt.title('Distribution of average throughput per handset (top 10 handsets)'); plt.show()

plt.figure(figsize=(12, 6))
sub.groupby('handset')['avg_tcp'].mean().loc[top_handsets].sort_values().plot.barh(color='indianred')
plt.title('Average TCP retransmission per handset (top 10 handsets)'); plt.show()

In [ ]:
handset_summary = (exp_known.groupby('handset')
                   .agg(users=('avg_tput', 'size'), mean_tput=('avg_tput', 'mean'), mean_tcp=('avg_tcp', 'mean')))
handset_summary = handset_summary[handset_summary['users'] >= 100]     # ignore rarely used handsets
display(handset_summary.sort_values('mean_tput', ascending=False).head(10))
display(handset_summary.sort_values('mean_tcp', ascending=False).head(10))

**Interpretation (write your own):** do some handsets get consistently better throughput or fewer retransmissions? Is it a device issue (modem/antenna) or a network issue?

### 3.4 k-means (k = 3) on experience metrics

In [ ]:
exp_features = ['avg_tcp', 'avg_rtt', 'avg_tput']
exp_scaler = StandardScaler()
X_exp = exp_scaler.fit_transform(exp[exp_features])

km_exp = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_exp)
exp['exp_cluster'] = km_exp.labels_

# Name clusters: experience index = throughput (good) minus TCP retrans and RTT (bad), in standardized units
centers_z = pd.DataFrame(km_exp.cluster_centers_, columns=exp_features)
centers_z['index'] = centers_z['avg_tput'] - centers_z['avg_tcp'] - centers_z['avg_rtt']
order = centers_z['index'].sort_values(ascending=False).index.tolist()
names = dict(zip(order, ['Best (by index)', 'Middle (by index)', 'Worst (by index)']))
exp['exp_cluster_name'] = exp['exp_cluster'].map(names)
worst_exp_cluster = order[-1]      # if the table below tells a different story, set this manually (0, 1 or 2)

display(exp.groupby('exp_cluster_name')[exp_features].mean())
display(exp['exp_cluster_name'].value_counts())

**Cluster descriptions (write your own):** read the table above and describe each group in plain words (e.g. 'very high TCP retransmission', 'low throughput but low RTT'). The 'by index' names are only a starting point; the cluster you treat as *worst experience* feeds Task 4, so check it makes sense.

# Task 4 – Satisfaction Analysis

### 4.1 Engagement score and experience score
* Engagement score = Euclidean distance from each user to the **centre of the least-engaged cluster** (from the first k=3 clustering, in normalized space).
* Experience score = Euclidean distance from each user to the **centre of the worst-experience cluster**.

In [ ]:
eng_centers = pd.DataFrame(km_eng.cluster_centers_, columns=['sessions', 'duration_ms', 'total_data'])
least_engaged = eng_centers.sum(axis=1).idxmin()      # lowest values on all metrics
eng['engagement_score'] = np.linalg.norm(X_eng - km_eng.cluster_centers_[least_engaged], axis=1)

exp['experience_score'] = np.linalg.norm(X_exp - km_exp.cluster_centers_[worst_exp_cluster], axis=1)

scores = eng[['engagement_score']].join(exp[['experience_score']], how='inner')
scores.describe()

### 4.2 Satisfaction score = average of both scores → top 10 satisfied customers

The two scores are on different scales (min-max vs standardized). To follow the task literally keep `NORMALIZE_SCORES = False`; set it to `True` if you want both scaled to 0–1 before averaging.

In [ ]:
NORMALIZE_SCORES = False
s = scores.copy()
if NORMALIZE_SCORES:
    s[['engagement_score', 'experience_score']] = MinMaxScaler().fit_transform(s[['engagement_score', 'experience_score']])
s['satisfaction_score'] = (s['engagement_score'] + s['experience_score']) / 2

top10_satisfied = s.nlargest(10, 'satisfaction_score')
top10_satisfied

### 4.3 Regression model to predict satisfaction score
Features = the raw engagement and experience metrics. Because the satisfaction score is *calculated* from these metrics, a high R² is expected – mention this in your limitations.

In [ ]:
model_df = user[['sessions', 'duration_ms', 'total_data']].join(exp[['avg_tcp', 'avg_rtt', 'avg_tput']]).join(s['satisfaction_score'])
X = model_df.drop(columns='satisfaction_score')
y = model_df['satisfaction_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(model, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {'model': name, 'R2': r2_score(y_test, pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, pred)), 'MAE': mean_absolute_error(y_test, pred)}

results = pd.DataFrame([
    evaluate(LinearRegression(), 'Linear Regression'),
    evaluate(RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42), 'Random Forest')])
results

### 4.4 k-means (k = 2) on engagement & experience scores

In [ ]:
X_scores = StandardScaler().fit_transform(s[['engagement_score', 'experience_score']])
km_sat = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_scores)
s['sat_cluster'] = km_sat.labels_

sns.scatterplot(data=s.sample(5000, random_state=1), x='engagement_score', y='experience_score',
                hue='sat_cluster', palette='Set1', s=15)
plt.title('k-means (k=2) on engagement & experience scores'); plt.show()

### 4.5 Average satisfaction & experience score per cluster

In [ ]:
s.groupby('sat_cluster')[['satisfaction_score', 'experience_score', 'engagement_score']].mean()

### 4.6 Export the final table to MySQL
Install once: `pip install sqlalchemy pymysql`. First create the database in MySQL: `CREATE DATABASE telecom;`

In [ ]:
final = s.reset_index()[['MSISDN', 'engagement_score', 'experience_score', 'satisfaction_score']]
final.to_csv('user_scores.csv', index=False)         # backup copy
final.head()

In [ ]:
from sqlalchemy import create_engine

DB_USER, DB_PASS, DB_HOST, DB_NAME = 'root', 'your_password', 'localhost', 'telecom'   # <-- edit
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}')

final.to_sql('user_satisfaction', engine, if_exists='replace', index=False)
pd.read_sql('SELECT * FROM user_satisfaction LIMIT 10', engine)     # take your screenshot of this output

### 4.7 Model tracking with MLflow
Install once: `pip install mlflow`. This logs code version (git commit), parameters, metrics, per-epoch loss (convergence), the model and output files. View it with `mlflow ui` in a terminal, then open http://127.0.0.1:5000 for screenshots.

In [ ]:
import subprocess, mlflow, mlflow.sklearn
from sklearn.linear_model import SGDRegressor

def git_commit():
    try: return subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
    except Exception: return 'not-a-git-repo'

sc = StandardScaler().fit(X_train)
Xtr, Xte = sc.transform(X_train), sc.transform(X_test)

mlflow.set_experiment('telecom_satisfaction')
with mlflow.start_run(run_name='sgd_regression'):
    params = {'model': 'SGDRegressor', 'alpha': 1e-4, 'eta0': 0.01, 'epochs': 30, 'normalize_scores': NORMALIZE_SCORES}
    mlflow.log_params(params)
    mlflow.set_tag('git_commit', git_commit())
    mlflow.set_tag('data_source', DATA_PATH)

    sgd = SGDRegressor(alpha=params['alpha'], eta0=params['eta0'], learning_rate='constant', random_state=42)
    for epoch in range(params['epochs']):                       # loss convergence
        sgd.partial_fit(Xtr, y_train)
        mlflow.log_metric('train_mse', mean_squared_error(y_train, sgd.predict(Xtr)), step=epoch)

    pred = sgd.predict(Xte)
    mlflow.log_metrics({'test_r2': r2_score(y_test, pred),
                        'test_rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
                        'test_mae': mean_absolute_error(y_test, pred)})
    mlflow.sklearn.log_model(sgd, 'model')
    mlflow.log_artifact('user_scores.csv')

runs = mlflow.search_runs(experiment_names=['telecom_satisfaction'])      # includes start/end time, params, metrics
runs.to_csv('mlflow_runs_report.csv', index=False)
runs.T.head(30)

# Final summary (fill in after running everything)
* **Growth potential:** positive / negative – justify with data (customer base, engagement mix, experience).
* **Recommendation:** buy / don't buy TellCo.
* **Limitations:** one month of data; outliers replaced by the mean; satisfaction score derived from the same features it is predicted from; 'Other' traffic is not a real app; unknown handsets.